# Traffic 192 · 336 · 720: 더 오래 학습하면 논문값이 나오는가

**목적.** 논문 당시 코드(2022-06 커밋 `04ffd8d`)로 Traffic을 돌리면 논문보다 MSE가 0.001~0.002 높다. 96 한 칸은 최대 epoch를 10에서 30으로 늘리자 논문과 정확히 일치했다(0.4103 / 0.2822, 16번째 epoch에서 조기 종료). 나머지 세 칸도 같은지 확인한다.

| T | 지금까지 (최대 10 epoch) | 논문 |
|---|---|---|
| 96 | 0.4112 / 0.2839 → 30 epoch에서 0.4103 / 0.2822 | 0.410 / 0.282 |
| 192 | 0.4245 / 0.2912 | 0.423 / 0.287 |
| 336 | 0.4375 / 0.2989 | 0.436 / 0.296 |
| 720 | 0.4680 / 0.3180 | 0.466 / 0.315 |

**판정 기준.** 세 칸 모두 MSE와 MAE가 소수 셋째 자리까지 논문과 같으면 원인 확인.

**실행.** 런타임 **GPU → T4** → **Run All**. 예상 시간 1~2시간. 끝나면 맨 아래 셀의 출력 한 덩어리를 복사해서 채팅에 붙여 주세요. (`/content/traffic_ep30.csv`도 만들어집니다.)

**바꾸는 것.** 논문 당시 코드는 그대로 두고 두 가지만 고친다.
1. `np.Inf` → `np.inf` (NumPy 2 호환, 결과 영향 없음)
2. 테스트 오차를 배치마다 누적해서 계산 (원래는 예측값 전체를 메모리에 모았다가 계산해서 Traffic 720에서 메모리가 부족함). 같은 값이 나오는 것을 CPU에서 확인했다(Traffic 96: 0.4112 / 0.2839).

In [ ]:
# 0. GPU 확인
!nvidia-smi --query-gpu=name,memory.total --format=csv
!free -g | head -2

In [ ]:
# 1. 코드와 데이터 준비 (이미 있으면 건너뜀)
import os, subprocess
os.chdir('/content')
if not os.path.exists('LTSF-Linear'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/cure-lab/LTSF-Linear.git'], check=True)
os.chdir('/content/LTSF-Linear')
OLD = '/content/LTSF-old'
if not os.path.exists(OLD):
    r = subprocess.run(['git', 'worktree', 'add', '-q', OLD, '04ffd8d'])
    if r.returncode != 0:
        subprocess.run(['git', 'fetch', '-q', '--unshallow'])
        subprocess.run(['git', 'worktree', 'add', '-q', OLD, '04ffd8d'], check=True)

def patch(path, old, new):
    s = open(path).read()
    if new in s:
        return
    assert old in s, (path, old[:60])
    open(path, 'w').write(s.replace(old, new))

patch(f'{OLD}/utils/tools.py', 'np.Inf', 'np.inf')

# 테스트 오차를 배치마다 누적 (float64)
patch(f'{OLD}/exp/exp_main.py',
"""                preds.append(pred)
                trues.append(true)
                inputx.append(batch_x.detach().cpu().numpy())
                if i % 20 == 0:""",
"""                _d = pred.astype(np.float64) - true.astype(np.float64)
                _se = locals().get('_se', 0.0) + float((_d ** 2).sum())
                _ae = locals().get('_ae', 0.0) + float(np.abs(_d).sum())
                _n = locals().get('_n', 0) + _d.size
                if False:""")
s = open(f'{OLD}/exp/exp_main.py').read()
a = s.index('        preds = np.array(preds)\n        trues = np.array(trues)')
b = s.index('        return\n', a)
s = s[:a] + "        print('mse:{}, mae:{}'.format(_se / _n, _ae / _n))\n" + s[b:]
open(f'{OLD}/exp/exp_main.py', 'w').write(s)

os.makedirs(f'{OLD}/dataset', exist_ok=True)
if not os.path.exists(f'{OLD}/dataset/traffic.csv'):
    subprocess.run('pip -q install gdown && gdown -q --folder '
                   'https://drive.google.com/drive/folders/1ZOYpTUa82_jCcxIdTmyr0LXQfvaM9vIy -O /content/raw_data && '
                   f'find /content/raw_data -name "traffic.csv" -exec cp -n {{}} {OLD}/dataset/ \;', shell=True, check=True)
!git -C /content/LTSF-old log --oneline -1
!grep -n "drop_last = True" /content/LTSF-old/data_provider/data_factory.py
!grep -n "Linear_Decoder = nn.Linear" /content/LTSF-old/models/DLinear.py
!grep -n "_se / _n" /content/LTSF-old/exp/exp_main.py
!ls -lh /content/LTSF-old/dataset/traffic.csv

## 2. 실행
2022-06 Traffic 스크립트와 같은 설정(L=336, 학습률 0.05, 배치 16, seed 2021)에 `--train_epochs 30`만 붙인다. 조기 종료(patience 3)는 그대로 둔다. 192 → 336 → 720 순서로 돌고, 이미 끝난 칸은 건너뛴다.

In [ ]:
import re, time, pandas as pd
os.chdir(OLD)
os.makedirs('/content/traffic_logs', exist_ok=True)
PAPER = {96: (0.410, 0.282), 192: (0.423, 0.287), 336: (0.436, 0.296), 720: (0.466, 0.315)}
TEN = {96: (0.4112, 0.2839), 192: (0.4245, 0.2912), 336: (0.4375, 0.2989), 720: (0.4680, 0.3180)}

def run(P):
    log = f'/content/traffic_logs/traffic_{P}_ep30.log'
    if os.path.exists(log) and re.search(r'^mse:', open(log).read(), re.M):
        return log
    cmd = (f'python -u run_longExp.py --is_training 1 --root_path ./dataset/ --data_path traffic.csv '
           f'--model_id traffic_336_{P}_ep30 --model DLinear --data custom --features M --seq_len 336 '
           f'--pred_len {P} --enc_in 862 --des Exp --itr 1 --batch_size 16 --learning_rate 0.05 '
           f'--train_epochs 30 --num_workers 2 > {log} 2>&1')
    t = time.time(); subprocess.run(cmd, shell=True)
    print(f'T={P} done in {(time.time()-t)/60:.1f} min'); subprocess.run('rm -rf checkpoints/traffic_*', shell=True)
    return log

rows = []
for P in [192, 336, 720]:
    s = open(run(P)).read()
    m = re.search(r'^mse:([0-9.]+), mae:([0-9.]+)', s, re.M)
    ep = re.findall(r'^Epoch: (\d+), Steps.*Test Loss: ([0-9.]+)', s, re.M)
    mse, mae = (float(m.group(1)), float(m.group(2))) if m else (None, None)
    rows.append(dict(pred_len=P, epochs_run=len(ep), mse=mse, mae=mae, paper_mse=PAPER[P][0], paper_mae=PAPER[P][1],
                     mse_10ep=TEN[P][0], mae_10ep=TEN[P][1], test_mse_by_epoch=' '.join(t[:6] for _, t in ep)))
    print(rows[-1])
df = pd.DataFrame(rows)
df['match'] = [(m is not None) and round(m + 1e-9, 3) == pm and round(a + 1e-9, 3) == pa
               for m, a, pm, pa in zip(df.mse, df.mae, df.paper_mse, df.paper_mae)]
df.to_csv('/content/traffic_ep30.csv', index=False)
df[['pred_len', 'epochs_run', 'mse', 'mae', 'paper_mse', 'paper_mae', 'mse_10ep', 'match']].round(4)

## 3. 결과를 채팅에 붙여 주세요
아래 셀 출력을 그대로 복사해 주시면 됩니다.

In [ ]:
print(open('/content/traffic_ep30.csv').read())